# Exercise: Automated Expense Report Processing with Generative AI

## Learning Objectives

In this exercise, you will:
1. Learn to programmatically call a Large Language Model (LLM) API using AWS Bedrock
2. Transform unstructured text data into structured, machine-readable formats
3. Apply few-shot prompting techniques to guide AI behavior
4. Implement business logic for automated decision-making
5. Understand how generative AI integrates into production business processes

## Business Context

Organizations process thousands of expense reports annually, requiring significant manual effort to review, categorize, and approve employee expenses. Traditional approaches involve employees submitting expense descriptions, which managers must manually review against company policy before approval.

This exercise demonstrates how generative AI can automate this workflow by:
- Extracting structured data from unstructured expense descriptions
- Applying business rules consistently
- Making approval decisions based on company policy
- Reducing processing time from minutes to seconds per expense

## Company Expense Policy

Use these business rules when you construct your prompt in the prompt.txt file

### Expense Categories and Limits

| Category | Maximum Amount | Business Justification Required |
|----------|----------------|----------------------------------|
| Client Entertainment | $500 | Yes |
| Travel - Airfare | $1,500 | Yes |
| Travel - Hotel | $300/night | Yes |
| Travel - Ground Transportation | $100 | No |
| Meals - Solo Business | $75 | Yes |
| Office Supplies | $200 | No |
| Professional Development | $1,000 | Yes |
| Technology/Equipment | $2,000 | Yes |

### Approval Rules

1. **AUTO-APPROVE**: Expense is within category limit AND includes business justification (if required)
2. **FLAG FOR REVIEW**: Expense exceeds category limit OR missing required business justification
3. **REJECT**: Expense is for non-business purposes or prohibited items (personal entertainment, alcohol without clients, etc.)

### Required Information

All approved expenses must include:
- Expense category
- Amount in USD
- Date of expense
- Business justification (where required)
- Vendor/merchant name

## Setup: Install Required Libraries

First, install the AWS SDK for Python (boto3) which enables programmatic access to AWS Bedrock.

**Note for GitHub Codespaces**: This notebook is fully compatible with GitHub Codespaces. All dependencies will be installed automatically.

In [16]:
# Install required libraries
!pip install boto3 python-dotenv requests -q

zsh:1: command not found: pip


## Configure AWS Credentials

Your AWS credentials will be loaded from the `.env` file in this directory. Your instructor will provide the credentials for you to add to that file.

**Security Note**: In production environments, credentials should never be hardcoded or committed to version control. Use environment variables, AWS IAM roles, or secure credential management systems like AWS Secrets Manager.

In [ ]:
import boto3
import json
import os
import requests
from datetime import datetime
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Verify credentials are loaded
if not os.getenv('BEDROCK_API_KEY'):
    raise ValueError("BEDROCK_API_KEY not found in .env file. Please add your credentials.")
if not os.getenv('AWS_REGION'):
    raise ValueError("AWS_REGION not found in .env file. Please add your region.")

# Store credentials for API calls
BEDROCK_API_KEY = os.getenv('BEDROCK_API_KEY')
AWS_REGION = os.getenv('AWS_REGION', 'us-east-1')

print("AWS Bedrock credentials loaded successfully")
print(f"Region: {AWS_REGION}")

## Select Which Expense to Process

Choose which expense file to analyze by changing the filename below. In production, this would be automated - the system would process expenses as they are submitted via webhooks or message queues. For this learning exercise, you'll process expenses one at a time to observe the results.

**Available expense files:**
- `expense_001.txt` - Client entertainment
- `expense_002.txt` - Business travel (airfare)
- `expense_003.txt` - Office equipment purchase
- `expense_004.txt` - Professional development
- `expense_005.txt` - Ground transportation

In [ ]:
# TODO: Change this filename to process different expenses
EXPENSE_FILENAME = "expense_001.txt"

def load_expense_description(filename):
    """
    Load an expense description from a text file.
    
    Args:
        filename (str): Name of the file in the 'sample_expenses' directory
    
    Returns:
        str: The expense description text
    """
    filepath = f"sample_expenses/{filename}"
    with open(filepath, 'r') as f:
        return f.read().strip()

# Load the selected expense
expense_description = load_expense_description(EXPENSE_FILENAME)
print(f"Selected Expense: {EXPENSE_FILENAME}")
print("=" * 80)
print(expense_description)
print("=" * 80)

## Task 1: Design Your Prompt Using Few-Shot Prompting

Your primary task in this exercise is to design an effective prompt using few-shot prompting. The prompt is stored in `prompt.txt` and will be loaded and used to analyze expenses.

### What is Few-Shot Prompting?

Few-shot prompting is a technique where you provide the LLM with examples of the desired input-output format. These examples teach the model:
- What kind of input to expect
- What structure the output should follow
- How to apply business rules
- What level of detail to include

### Your Prompt Should Include:

1. **System Instructions**: Explain the LLM's role as an automated expense processing system
2. **Company Policy Rules**: Clearly state all expense categories, limits, and approval rules
3. **Few-Shot Examples (2-3)**: Show example expense descriptions and their expected JSON outputs
   - Include an AUTO-APPROVE example
   - Include a FLAG FOR REVIEW example  
   - Consider including a REJECT example or edge case
4. **Output Format**: Specify the exact JSON structure required
5. **The Actual Expense**: The specific expense description to analyze (this will be inserted automatically)

### Required Output Format (JSON):


```json
{
  "category": "text",
  "amount": "number",
  "date": "date",
  "vendor": "text",
  "business_justification": "text",
  "decision": "AUTO-APPROVE | FLAG FOR REVIEW | REJECT",
  "reasoning": "string explaining the decision"
}
```

**Now open `prompt.txt` and design your prompt.** Once you've created your prompt, continue to the next cell to test it.

In [15]:
def load_prompt_template():
    """
    Load the prompt template from prompt.txt
    
    Returns:
        str: The prompt template
    """
    with open('prompt.txt', 'r') as f:
        return f.read()

# Load your prompt template
prompt_template = load_prompt_template()

print("✓ Prompt template loaded successfully")
print(f"✓ Prompt length: {len(prompt_template)} characters")
print("\nPrompt Preview (first 500 characters):")
print("=" * 80)
print(prompt_template[:500])
print("...")
print("=" * 80)

✓ Prompt template loaded successfully
✓ Prompt length: 2593 characters

Prompt Preview (first 500 characters):

SYSTEM INSTRUCTIONS:
[INSTRUCTIONS - DELETE BEFORE RUNNING PROMPT: provide system instructions - what are you asking the LLM to do? e.g. You are a customer service agent. Your task is to do XYZ]


COMPANY EXPENSE POLICIES:
[INSTRUCTIONS - DELETE BEFORE RUNNING PROMPT: List all 8 expense categories with their maximum amounts and justification requirements from the notebook's policy table]


DECISION RULES:
[INSTRUCTIONS - DELETE BEFORE RUNNING PROMPT: Define the criteria for when to AUTO-APPROVE
...


## Task 2: Call the LLM API

Now we'll send your prompt to AWS Bedrock and get a response. This cell handles all the API interaction.

### Understanding the API Call

**Model Selection**: This exercise uses `anthropic.claude-3-haiku-20240307-v1:0`

Why Claude 3 Haiku?
- **Speed**: Processes requests in 2-3 seconds (vs. 10+ seconds for larger models)
- **Cost**: $0.25 per million input tokens, $1.25 per million output tokens (very inexpensive)
- **Capability**: Excellent for structured data extraction tasks like this
- **Appropriate Use Case**: When you need fast, consistent outputs for well-defined tasks

When might you choose a different model?
- **Claude 3 Sonnet**: More complex reasoning, nuanced understanding, multi-step analysis
- **Claude 3 Opus**: Highest capability, creative tasks, ambiguous situations requiring judgment

**Temperature Setting**: Set to 0.0
- Temperature controls randomness in the model's output
- 0.0 = Most deterministic, consistent outputs (best for structured data extraction)
- Higher values (0.7-1.0) = More creative, varied outputs (better for content generation)

**Max Tokens**: Set to 1000
- Maximum number of tokens the model can generate in response
- Our JSON responses are typically 200-300 tokens, so 1000 provides a safe buffer

In [ ]:
def analyze_expense_with_llm(expense_description, prompt_template):
    """
    Send expense description to AWS Bedrock for analysis.
    
    Args:
        expense_description (str): The unstructured expense text
        prompt_template (str): The prompt template from prompt.txt
    
    Returns:
        tuple: (parsed_result dict, raw_response_metadata dict)
    """
    
    # Insert the expense description into the prompt template
    # The template should have a placeholder like {EXPENSE_DESCRIPTION}
    full_prompt = prompt_template.replace("{EXPENSE_DESCRIPTION}", expense_description)
    
    # Prepare the request body for Claude 3 Haiku
    # This follows the Anthropic Messages API format required by Bedrock
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",  # API version
        "max_tokens": 1000,  # Maximum response length
        "messages": [
            {
                "role": "user",  # We're sending a user message
                "content": full_prompt  # Our complete prompt with examples and expense
            }
        ],
        "temperature": 0.0,  # Deterministic output for consistency
        "top_p": 1.0  # Use all possible tokens (no nucleus sampling)
    }
    
    # Call the Bedrock API using Bearer token authentication
    url = f"https://bedrock-runtime.{AWS_REGION}.amazonaws.com/model/anthropic.claude-3-haiku-20240307-v1:0/invoke"
    headers = {
        "Authorization": f"Bearer {BEDROCK_API_KEY}",
        "Content-Type": "application/json"
    }
    
    response = requests.post(url, headers=headers, json=request_body, timeout=30)
    
    # Check for errors
    if response.status_code != 200:
        raise Exception(f"Bedrock API error: {response.status_code} - {response.text}")
    
    # Parse the response from Bedrock
    response_body = response.json()
    
    # Store raw response metadata for analysis
    raw_metadata = {
        "model": response_body.get('model', 'anthropic.claude-3-haiku-20240307-v1:0'),
        "stop_reason": response_body.get('stop_reason'),
        "usage": response_body.get('usage', {}),
        "full_response": response_body
    }
    
    llm_output = response_body['content'][0]['text']
    
    # Extract JSON from the response
    # The LLM might wrap JSON in markdown code blocks (```json ... ```), so we clean it
    if '```json' in llm_output:
        llm_output = llm_output.split('```json')[1].split('```')[0].strip()
    elif '```' in llm_output:
        llm_output = llm_output.split('```')[1].split('```')[0].strip()
    
    # Parse the JSON response into a Python dictionary
    result = json.loads(llm_output)
    
    return result, raw_metadata

# Analyze the selected expense
print(f"Analyzing {EXPENSE_FILENAME}...\n")
result, raw_metadata = analyze_expense_with_llm(expense_description, prompt_template)

# Display the structured result
print("ANALYSIS RESULT:")
print("=" * 80)
print(json.dumps(result, indent=2))
print("=" * 80)

## Understanding API Response and Token Usage

When you call an LLM API, the response includes important metadata beyond just the generated text. Understanding this metadata helps you:
- Calculate actual costs per API call
- Optimize prompt design for efficiency
- Monitor performance and identify bottlenecks
- Debug issues when they occur

### Token Usage Explained

**What is a Token?**
Tokens are the basic units that LLMs process. A token is roughly equivalent to:
- 1 token = ~4 characters of English text
- 1 token = ~0.75 words on average
- Example: "Hello, world!" = 4 tokens

**Why Tokens Matter:**
1. **Cost**: You pay per token (input + output)
2. **Limits**: Models have maximum context windows (e.g., 200,000 tokens for Claude 3 Haiku)
3. **Performance**: More tokens = longer processing time

### API Response Structure

The raw API response contains several key fields:

**usage** (token counts):
- `input_tokens`: Number of tokens in your prompt (instructions + examples + expense description)
- `output_tokens`: Number of tokens the model generated in its response
- Total cost = (input_tokens × input_price) + (output_tokens × output_price)

**stop_reason**: Why the model stopped generating
- `end_turn`: Natural completion (model finished its response)
- `max_tokens`: Hit the max_tokens limit (response was cut off)
- `stop_sequence`: Encountered a specified stop sequence

**model**: The specific model version used (important for reproducibility)

**content**: The actual generated text (in our case, JSON)

Let's examine the raw response from the API call above.

In [ ]:
# Display raw API response metadata
print("RAW API RESPONSE METADATA")
print("=" * 80)
print(f"Model: {raw_metadata['model']}")
print(f"Stop Reason: {raw_metadata['stop_reason']}")
print()
print("TOKEN USAGE:")
print(f"  Input Tokens:  {raw_metadata['usage']['input_tokens']:,}")
print(f"  Output Tokens: {raw_metadata['usage']['output_tokens']:,}")
print(f"  Total Tokens:  {raw_metadata['usage']['input_tokens'] + raw_metadata['usage']['output_tokens']:,}")
print()

# Calculate cost for this API call
# Claude 3 Haiku pricing (as of 2024)
INPUT_PRICE_PER_MILLION = 0.25   # $0.25 per million input tokens
OUTPUT_PRICE_PER_MILLION = 1.25  # $1.25 per million output tokens

input_cost = (raw_metadata['usage']['input_tokens'] / 1_000_000) * INPUT_PRICE_PER_MILLION
output_cost = (raw_metadata['usage']['output_tokens'] / 1_000_000) * OUTPUT_PRICE_PER_MILLION
total_cost = input_cost + output_cost

print("COST BREAKDOWN:")
print(f"  Input Cost:  ${input_cost:.6f}")
print(f"  Output Cost: ${output_cost:.6f}")
print(f"  Total Cost:  ${total_cost:.6f}")
print()
print(f"Cost per expense: ${total_cost:.4f}")
print(f"Projected cost for 1,000 expenses: ${total_cost * 1000:.2f}")
print(f"Projected cost for 10,000 expenses: ${total_cost * 10000:.2f}")
print("=" * 80)

# Optional: Display full raw response for debugging
print("\nFULL RAW RESPONSE (for debugging):")
print("=" * 80)
print(json.dumps(raw_metadata['full_response'], indent=2))
print("=" * 80)

## Optimizing Token Usage and Cost

Based on the token usage above, consider these optimization strategies:

### 1. Prompt Efficiency
**Current Approach**: Your prompt includes system instructions, company policy, few-shot examples, and the expense description.

**Optimization Strategies**:
- **Reduce Example Count**: Start with 3 examples, test if 2 examples maintain accuracy
- **Compress Policy Text**: Use concise language without sacrificing clarity
- **Remove Redundancy**: Eliminate repeated information across instructions and examples
- **Use Abbreviations**: Define abbreviations for repeated terms (e.g., "BJ" for "Business Justification")

**Trade-off**: Shorter prompts save money but may reduce accuracy. Test thoroughly.

### 2. Batch Processing
**Current Approach**: One API call per expense (demonstrated here for learning purposes).

**Production Optimization**: Process multiple expenses in a single API call:
- Provide all expense descriptions in one prompt
- Request a JSON array response with all results
- Reduces overhead from multiple API calls
- More efficient token usage (shared policy/instructions)

**Example**: Processing 10 expenses separately costs more than processing all 10 in one call.

### 3. Model Selection
**Claude 3 Haiku**: Fast, inexpensive, excellent for structured tasks
- Input: $0.25 per million tokens
- Output: $1.25 per million tokens
- Best for: High-volume, well-defined tasks

**Claude 3 Sonnet**: Balanced performance
- Input: $3 per million tokens (12x more expensive)
- Output: $15 per million tokens (12x more expensive)
- Best for: Complex reasoning, nuanced judgment

**Claude 3 Opus**: Highest capability
- Input: $15 per million tokens (60x more expensive)
- Output: $75 per million tokens (60x more expensive)
- Best for: Ambiguous cases, creative tasks, highest accuracy requirements

**Strategy**: Use Haiku for most expenses, escalate to Sonnet/Opus only for flagged cases.

### 4. Caching and Reuse
**Opportunity**: The policy rules and few-shot examples are identical across all expenses.

**Advanced Optimization** (requires additional API features):
- Use prompt caching to store static portions of the prompt
- Only send the variable expense description each time
- Can reduce input tokens by 50-70% for repeated prompts

### 5. Output Format Optimization
**Current Approach**: Verbose JSON with field names and detailed reasoning.

**Alternative**: Use shorter field names, request more concise reasoning.

**Trade-off**: Sacrifices readability for cost savings. Usually not worth it.



## Task 3: Interpret the Results

Review the analysis result above and verify:

1. **Correct Categorization**: Did the LLM assign the right expense category?
2. **Accurate Extraction**: Are the amount, date, and vendor correctly identified?
3. **Appropriate Decision**: Based on the policy, is the AUTO-APPROVE/FLAG/REJECT decision correct?
4. **Sound Reasoning**: Does the reasoning clearly explain why the decision was made?

### Testing Different Expenses

Go back to the "Select Which Expense to Process" section and change the `EXPENSE_FILENAME` variable to test different expenses:
- `expense_001.txt`
- `expense_002.txt`
- `expense_003.txt`
- `expense_004.txt`
- `expense_005.txt`

Then re-run the cells from that point forward to see how your prompt handles different scenarios.

**Note**: In production, this manual process would be automated. Each time an employee submits an expense, the system would automatically:
1. Receive the expense via webhook or message queue
2. Call the LLM API with the expense description
3. Store the structured result in a database
4. Route the expense based on the decision (auto-approve, send to manager, etc.)
5. Update the employee dashboard in real-time

In [ ]:
# Optional: Process all expenses at once to generate a summary report
# This is useful for evaluating your prompt's performance across different scenarios

expense_files = [
    "expense_001.txt",
    "expense_002.txt",
    "expense_003.txt",
    "expense_004.txt",
    "expense_005.txt"
]

results = []
total_input_tokens = 0
total_output_tokens = 0

print("Processing all expenses...\n")

for filename in expense_files:
    print(f"Processing {filename}...")
    
    # Load expense description
    expense_desc = load_expense_description(filename)
    
    # Analyze with LLM (now returns tuple: result, metadata)
    result, metadata = analyze_expense_with_llm(expense_desc, prompt_template)
    result['filename'] = filename
    results.append(result)
    
    # Accumulate token usage
    total_input_tokens += metadata['usage']['input_tokens']
    total_output_tokens += metadata['usage']['output_tokens']
    
    # Display brief result
    print(f"  Category: {result['category']}")
    print(f"  Amount: ${result['amount']:.2f}")
    print(f"  Decision: {result['decision']}")
    print(f"  Tokens: {metadata['usage']['input_tokens'] + metadata['usage']['output_tokens']}")
    print()

print("=" * 80)
print("All expenses processed successfully!")
print("=" * 80)
print(f"\nTotal Token Usage Across All Expenses:")
print(f"  Input Tokens:  {total_input_tokens:,}")
print(f"  Output Tokens: {total_output_tokens:,}")
print(f"  Total Tokens:  {total_input_tokens + total_output_tokens:,}")

# Calculate total cost
total_batch_cost = ((total_input_tokens / 1_000_000) * 0.25) + ((total_output_tokens / 1_000_000) * 1.25)
print(f"\nTotal Cost: ${total_batch_cost:.6f}")
print(f"Average Cost Per Expense: ${total_batch_cost / len(expense_files):.6f}")
print("=" * 80)

## Summary Report

This report shows aggregate statistics across all processed expenses.

In [ ]:
from collections import Counter

# Calculate summary statistics
decisions = [r['decision'] for r in results]
decision_counts = Counter(decisions)

total_amount = sum(r['amount'] for r in results)
approved_amount = sum(r['amount'] for r in results if r['decision'] == 'AUTO-APPROVE')

print("\n" + "=" * 80)
print("EXPENSE PROCESSING SUMMARY REPORT")
print("=" * 80)
print(f"\nTotal Expenses Processed: {len(results)}")
print(f"Total Amount: ${total_amount:.2f}")
print(f"\nDecision Breakdown:")
for decision, count in decision_counts.items():
    print(f"  {decision}: {count}")

print(f"\nAuto-Approved Amount: ${approved_amount:.2f}")
print(f"Approval Rate: {(decision_counts.get('AUTO-APPROVE', 0) / len(results) * 100):.1f}%")

print("\n" + "-" * 80)
print("DETAILED RESULTS")
print("-" * 80)

for result in results:
    print(f"\n{result['filename']}:")
    print(f"  Category: {result['category']}")
    print(f"  Amount: ${result['amount']:.2f}")
    print(f"  Vendor: {result['vendor']}")
    print(f"  Decision: {result['decision']}")
    print(f"  Reasoning: {result['reasoning']}")

## Web Demo

This exercise includes a web interface (`app.py`) that demonstrates how the same expense analysis logic works in a website user interface that's integrated with the LLM API. The Flask application provides:

- Interactive form for submitting expenses
- Real-time analysis with visual feedback
- Token usage and cost display
- Sample expense selector

Run the cell below to start the web server.

BE SURE TO STOP THE SERVER BY CLICKING THE SQUARE ICON IN THE TOP LEFT OF THE CELL WHEN DONE

In [ ]:
# Start the Flask web application
# This will start a background process - you'll see output in the notebook

import subprocess
import time
import sys

print("Starting Flask web server...")
print("="*80)
print("\nThe server will start on http://localhost:5001")
print("\nTo access the web interface:")
print("  • Local: Open http://localhost:5001 in your browser")
print("  • Codespaces: Click the popup to open the forwarded port")
print("\nPress the STOP button above to shut down the server.")
print("="*80)
print()

# Start Flask app
subprocess.run([sys.executable, "app.py"])

## Reflection Questions

After completing the exercise, write a brief reflection (250-500 words total) addressing the following questions:

### 1. Few-Shot Prompting vs. Zero-Shot Prompting
How do you think few-shot prompting (providing examples) makes the output more accurate compared to zero-shot prompting (just giving instructions without examples)? What did you learn about the importance of example selection?

### 2. Prompt Engineering Insights
What did you learn about designing effective prompts? Were there any surprises in how the LLM interpreted your instructions or examples?

### 3. Model Selection Considerations
What did you learn about choosing the right model for a task? When would you consider using a more powerful (and expensive) model like Claude 3 Sonnet or Opus instead of Haiku?

### 4. Return on Investment (ROI)
How might you calculate the Return on Investment (ROI) for implementing this automated expense processing system? What factors would you consider in the cost-benefit analysis?

### 5. Process Improvements
Can you think of any ways to further improve this automated expense approval process? Consider accuracy, speed, user experience, or additional features that could add value.